# Voting in Authoritarian Elections

**Authors:** Turkuler Isiksel, Thomas B. Pepinsky

**Journal:** American Political Science Review (2026)

This notebook reproduces the analysis from the paper above.
It was auto-generated by [PoliRep](https://polirep.org).

---

## Setup & Data Loading

Load datasets from BigQuery. You need a GCP project for billing — the dataset itself is public.

Set `PROJECT_ID` to your own GCP project ID below.

In [ ]:
# Install required packages (uncomment if needed)
# !pip install pandas-gbq pandas

import pandas as pd

# Set your GCP project ID for billing
PROJECT_ID = "polirep-app"  # Change this to your GCP project

In [ ]:
# Load GWF Autocratic Regimes.csv
gwf_autocratic_regimes = pd.read_gbq(
    "SELECT * FROM `polirep-app.paper_data.replication_data_for_voting_in_authoritarian_elections_gwf_autocratic_regimes`",
    project_id=PROJECT_ID
)
print(f"gwf_autocratic_regimes: {gwf_autocratic_regimes.shape[0]} rows, {gwf_autocratic_regimes.shape[1]} columns")
gwf_autocratic_regimes.head()

In [ ]:
# Load GWF_AllPoliticalRegimes.parquet
gwf_allpoliticalregimes = pd.read_gbq(
    "SELECT * FROM `polirep-app.paper_data.replication_data_for_voting_in_authoritarian_elections_gwf_allpoliticalregimes`",
    project_id=PROJECT_ID
)
print(f"gwf_allpoliticalregimes: {gwf_allpoliticalregimes.shape[0]} rows, {gwf_allpoliticalregimes.shape[1]} columns")
gwf_allpoliticalregimes.head()

In [ ]:
# Load GWFcases.parquet
gwfcases = pd.read_gbq(
    "SELECT * FROM `polirep-app.paper_data.replication_data_for_voting_in_authoritarian_elections_gwfcases`",
    project_id=PROJECT_ID
)
print(f"gwfcases: {gwfcases.shape[0]} rows, {gwfcases.shape[1]} columns")
gwfcases.head()

In [ ]:
# Load GWFtscs.parquet
gwftscs = pd.read_gbq(
    "SELECT * FROM `polirep-app.paper_data.replication_data_for_voting_in_authoritarian_elections_gwftscs`",
    project_id=PROJECT_ID
)
print(f"gwftscs: {gwftscs.shape[0]} rows, {gwftscs.shape[1]} columns")
gwftscs.head()

In [ ]:
# Load institutions in dictatorships, 1946-2008.csv
institutions_in_dictatorships__1946_2008 = pd.read_gbq(
    "SELECT * FROM `polirep-app.paper_data.replication_data_for_voting_in_authoritarian_elections_institutions_in_dictatorships__1946_2008`",
    project_id=PROJECT_ID
)
print(f"institutions_in_dictatorships__1946_2008: {institutions_in_dictatorships__1946_2008.shape[0]} rows, {institutions_in_dictatorships__1946_2008.shape[1]} columns")
institutions_in_dictatorships__1946_2008.head()

In [ ]:
# Load institutions in dictatorships, 1946-2009.parquet
institutions_in_dictatorships__1946_2009 = pd.read_gbq(
    "SELECT * FROM `polirep-app.paper_data.replication_data_for_voting_in_authoritarian_elections_institutions_in_dictatorships__1946_2009`",
    project_id=PROJECT_ID
)
print(f"institutions_in_dictatorships__1946_2009: {institutions_in_dictatorships__1946_2009.shape[0]} rows, {institutions_in_dictatorships__1946_2009.shape[1]} columns")
institutions_in_dictatorships__1946_2009.head()

## Data Preparation


### Load GWF Cases Data

Load the regime case-level data from GWFcases. This dataset contains one
row per autocratic regime spell with start/end dates and regime characteristics.

In [ ]:
def load_main_data() -> pd.DataFrame:
    """Load the main analysis datasets from converted data.

    This paper uses multiple datasets:
    - GWFcases: Regime case-level data (for clean.py transformation)
    - GWF_AllPoliticalRegimes: All political regimes TSCS data (for figure1.py)
    - institutions in dictatorships: Svolik data on authoritarian institutions (for figure1.py)

    Returns:
        Dictionary with keys 'gwfcases', 'gwf_all', 'svolik'.
    """
    # Load GWFcases (regime case-level data)
    gwfcases = pd.read_parquet(DATA_CONVERTED / "GWFcases.parquet")

    # Load GWF All Political Regimes (TSCS data)
    gwf_all = pd.read_parquet(DATA_CONVERTED / "GWF_AllPoliticalRegimes.parquet")

    # Load Svolik institutions data (1946-2008 version)
    svolik = pd.read_parquet(DATA_CONVERTED / "institutions in dictatorships, 1946-2008.parquet")

    return {
        "gwfcases": gwfcases,
        "gwf_all": gwf_all,
        "svolik": svolik,
    }

In [ ]:
data_dict = load_main_data()
gwfcases = data_dict['gwfcases']
print(f"GWFcases shape: {gwfcases.shape}")
print(f"Regime types: {gwfcases['gwf_regimetype'].value_counts()}")
print(f"Year range: {int(gwfcases['gwf_startyr'].min())}-{int(gwfcases['gwf_endyr'].max())}")
print(f"Missing end year (censored): {gwfcases['gwf_endyr'].isna().sum()}")
display(gwfcases.head())


### Expand Cases to TSCS Format

Transform GWFcases from case-level to time-series cross-sectional format
by expanding each regime case into one row per year in power. Creates
sequential year variable, duration, and failure indicators.

In [ ]:
def run(gwfcases: pd.DataFrame) -> pd.DataFrame:
    """Transform GWFcases to time-series cross-sectional (TSCS) format.

    Args:
        gwfcases: Regime case-level DataFrame from load_main_data().

    Returns:
        GWFtscs DataFrame with one row per country-year.
    """
    df = gwfcases.copy()

    # Step 1: Create temporal variables
    df["byear"] = df["gwf_startyr"]
    df["eyear"] = df["gwf_endyr"].fillna(2010)  # Right-censoring
    df["spell"] = (df["eyear"] - df["byear"] + 1).astype(int)

    # Step 2: Create regime code (group ID)
    df["regimecode"] = df.groupby("gwf_casename").ngroup()

    # Step 3: Expand to TSCS format (one row per year)
    df_expanded = df.loc[df.index.repeat(df["spell"])].copy()

    # Step 4: Create year variable (sequential within regime)
    df_expanded["year"] = (
        df_expanded.groupby("regimecode").cumcount() + df_expanded["byear"]
    ).astype(int)

    # Step 5: Create duration and failure variables
    df_expanded["duration"] = df_expanded["year"] - (df_expanded["byear"] - 1)
    df_expanded["failure"] = (
        (df_expanded["year"] == df_expanded["gwf_endyr"])
        & (df_expanded["gwf_endyr"].notna())
    ).astype(int)

    # Step 6: Create failure type variables
    df_expanded["gwf_fail"] = df_expanded["failure"]
    df_expanded["gwf_fail_type"] = 0.0
    df_expanded.loc[df_expanded["failure"] == 1, "gwf_fail_type"] = df_expanded.loc[
        df_expanded["failure"] == 1, "gwf_howend"
    ]

    df_expanded["gwf_fail_subsregime"] = df_expanded["gwf_fail"].astype(float)
    df_expanded["gwf_fail_subs"] = 0
    df_expanded.loc[df_expanded["failure"] == 1, "gwf_fail_subs"] = df_expanded.loc[
        df_expanded["failure"] == 1, "gwf_subsreg"
    ]

    df_expanded["gwf_fail_violent"] = df_expanded["gwf_fail"].astype(float)
    df_expanded.loc[df_expanded["failure"] == 1, "gwf_fail_violent"] = df_expanded.loc[
        df_expanded["failure"] == 1, "gwf_violent"
    ]

    # Step 7: Calculate past regime failures
    # Sort by cowcode and year
    df_expanded = df_expanded.sort_values(["cowcode", "year"]).reset_index(drop=True)

    # Calculate cumulative failures within country, then shift by 1 (lag)
    df_expanded["gwf_pastregimefail"] = (
        df_expanded.groupby("cowcode")["gwf_fail"].cumsum().shift(1).fillna(0).astype(int)
    )

    # Step 8: Create collapsed regime type dummies
    # gwf_party: party, party-military, party-personal, party-personal-military, oligarchy, Iran post-1980
    party_types = [
        "party",
        "party-military",
        "party-personal",
        "party-personal-military",
        "oligarchy",
    ]
    df_expanded["gwf_party"] = df_expanded["gwf_regimetype"].isin(party_types).astype(int)

    # Handle Iran post-1980 (cowcode 630)
    iran_post_1980 = (df_expanded["cowcode"] == 630) & (df_expanded["year"] > 1980)
    df_expanded.loc[iran_post_1980, "gwf_party"] = 1

    # gwf_personal: personal regime
    df_expanded["gwf_personal"] = (df_expanded["gwf_regimetype"] == "personal").astype(float)

    # gwf_monarch: monarchy
    df_expanded["gwf_monarch"] = (df_expanded["gwf_regimetype"] == "monarchy").astype(float)

    # gwf_military: military, military-personal, indirect military
    military_types = ["military", "military-personal", "indirect military"]
    df_expanded["gwf_military"] = df_expanded["gwf_regimetype"].isin(military_types).astype(int)

    # Step 9: Filter and select key variables
    df_tscs = df_expanded[df_expanded["year"] > 1945].copy()

    # Select key variables (21 columns from analysis plan)
    key_vars = [
        "cowcode",
        "year",
        "gwf_casename",
        "gwf_regimetype",
        "gwf_startyr",
        "gwf_endyr",
        "byear",
        "eyear",
        "spell",
        "duration",
        "failure",
        "gwf_fail",
        "gwf_fail_type",
        "gwf_fail_subsregime",
        "gwf_fail_subs",
        "gwf_fail_violent",
        "gwf_pastregimefail",
        "gwf_party",
        "gwf_personal",
        "gwf_monarch",
        "gwf_military",
    ]

    df_tscs = df_tscs[key_vars].reset_index(drop=True)

    return df_tscs

In [ ]:
data_dict = load_main_data()
gwf_tscs = run(data_dict['gwfcases'])
print(f"GWFtscs shape: {gwf_tscs.shape}")
print(f"Countries: {gwf_tscs['cowcode'].nunique()}")
print(f"Year range: {gwf_tscs['year'].min()}-{gwf_tscs['year'].max()}")
print(f"Total failures: {gwf_tscs['gwf_fail'].sum()}")
print(f"Duration range: {gwf_tscs['duration'].min()}-{gwf_tscs['duration'].max()} years")
display(gwf_tscs[['cowcode', 'year', 'gwf_regimetype', 'duration', 'gwf_fail']].head(10))


### Create Regime Type Dummies

Create collapsed binary indicators for party, personal, military, and
monarchy regimes. Includes special handling for Iran post-1980 (coded
as party regime despite being a theocracy).

In [ ]:
def run(gwfcases: pd.DataFrame) -> pd.DataFrame:
    """Transform GWFcases to time-series cross-sectional (TSCS) format.

    Args:
        gwfcases: Regime case-level DataFrame from load_main_data().

    Returns:
        GWFtscs DataFrame with one row per country-year.
    """
    df = gwfcases.copy()

    # Step 1: Create temporal variables
    df["byear"] = df["gwf_startyr"]
    df["eyear"] = df["gwf_endyr"].fillna(2010)  # Right-censoring
    df["spell"] = (df["eyear"] - df["byear"] + 1).astype(int)

    # Step 2: Create regime code (group ID)
    df["regimecode"] = df.groupby("gwf_casename").ngroup()

    # Step 3: Expand to TSCS format (one row per year)
    df_expanded = df.loc[df.index.repeat(df["spell"])].copy()

    # Step 4: Create year variable (sequential within regime)
    df_expanded["year"] = (
        df_expanded.groupby("regimecode").cumcount() + df_expanded["byear"]
    ).astype(int)

    # Step 5: Create duration and failure variables
    df_expanded["duration"] = df_expanded["year"] - (df_expanded["byear"] - 1)
    df_expanded["failure"] = (
        (df_expanded["year"] == df_expanded["gwf_endyr"])
        & (df_expanded["gwf_endyr"].notna())
    ).astype(int)

    # Step 6: Create failure type variables
    df_expanded["gwf_fail"] = df_expanded["failure"]
    df_expanded["gwf_fail_type"] = 0.0
    df_expanded.loc[df_expanded["failure"] == 1, "gwf_fail_type"] = df_expanded.loc[
        df_expanded["failure"] == 1, "gwf_howend"
    ]

    df_expanded["gwf_fail_subsregime"] = df_expanded["gwf_fail"].astype(float)
    df_expanded["gwf_fail_subs"] = 0
    df_expanded.loc[df_expanded["failure"] == 1, "gwf_fail_subs"] = df_expanded.loc[
        df_expanded["failure"] == 1, "gwf_subsreg"
    ]

    df_expanded["gwf_fail_violent"] = df_expanded["gwf_fail"].astype(float)
    df_expanded.loc[df_expanded["failure"] == 1, "gwf_fail_violent"] = df_expanded.loc[
        df_expanded["failure"] == 1, "gwf_violent"
    ]

    # Step 7: Calculate past regime failures
    # Sort by cowcode and year
    df_expanded = df_expanded.sort_values(["cowcode", "year"]).reset_index(drop=True)

    # Calculate cumulative failures within country, then shift by 1 (lag)
    df_expanded["gwf_pastregimefail"] = (
        df_expanded.groupby("cowcode")["gwf_fail"].cumsum().shift(1).fillna(0).astype(int)
    )

    # Step 8: Create collapsed regime type dummies
    # gwf_party: party, party-military, party-personal, party-personal-military, oligarchy, Iran post-1980
    party_types = [
        "party",
        "party-military",
        "party-personal",
        "party-personal-military",
        "oligarchy",
    ]
    df_expanded["gwf_party"] = df_expanded["gwf_regimetype"].isin(party_types).astype(int)

    # Handle Iran post-1980 (cowcode 630)
    iran_post_1980 = (df_expanded["cowcode"] == 630) & (df_expanded["year"] > 1980)
    df_expanded.loc[iran_post_1980, "gwf_party"] = 1

    # gwf_personal: personal regime
    df_expanded["gwf_personal"] = (df_expanded["gwf_regimetype"] == "personal").astype(float)

    # gwf_monarch: monarchy
    df_expanded["gwf_monarch"] = (df_expanded["gwf_regimetype"] == "monarchy").astype(float)

    # gwf_military: military, military-personal, indirect military
    military_types = ["military", "military-personal", "indirect military"]
    df_expanded["gwf_military"] = df_expanded["gwf_regimetype"].isin(military_types).astype(int)

    # Step 9: Filter and select key variables
    df_tscs = df_expanded[df_expanded["year"] > 1945].copy()

    # Select key variables (21 columns from analysis plan)
    key_vars = [
        "cowcode",
        "year",
        "gwf_casename",
        "gwf_regimetype",
        "gwf_startyr",
        "gwf_endyr",
        "byear",
        "eyear",
        "spell",
        "duration",
        "failure",
        "gwf_fail",
        "gwf_fail_type",
        "gwf_fail_subsregime",
        "gwf_fail_subs",
        "gwf_fail_violent",
        "gwf_pastregimefail",
        "gwf_party",
        "gwf_personal",
        "gwf_monarch",
        "gwf_military",
    ]

    df_tscs = df_tscs[key_vars].reset_index(drop=True)

    return df_tscs

In [ ]:
data_dict = load_main_data()
gwf_tscs = run(data_dict['gwfcases'])
print(f"Party regimes: {gwf_tscs['gwf_party'].sum()} obs ({gwf_tscs['gwf_party'].mean():.1%})")
print(f"Personal regimes: {gwf_tscs['gwf_personal'].sum()} obs ({gwf_tscs['gwf_personal'].mean():.1%})")
print(f"Military regimes: {gwf_tscs['gwf_military'].sum()} obs ({gwf_tscs['gwf_military'].mean():.1%})")
print(f"Monarchies: {gwf_tscs['gwf_monarch'].sum()} obs ({gwf_tscs['gwf_monarch'].mean():.1%})")
display(gwf_tscs[['gwf_regimetype', 'gwf_party', 'gwf_personal', 'gwf_military', 'gwf_monarch']].head(10))


### Calculate Past Regime Failures

Compute cumulative count of prior regime failures within each country.
Uses lag(cumsum(gwf_fail)) grouped by country code to track regime
instability history.

In [ ]:
def run(gwfcases: pd.DataFrame) -> pd.DataFrame:
    """Transform GWFcases to time-series cross-sectional (TSCS) format.

    Args:
        gwfcases: Regime case-level DataFrame from load_main_data().

    Returns:
        GWFtscs DataFrame with one row per country-year.
    """
    df = gwfcases.copy()

    # Step 1: Create temporal variables
    df["byear"] = df["gwf_startyr"]
    df["eyear"] = df["gwf_endyr"].fillna(2010)  # Right-censoring
    df["spell"] = (df["eyear"] - df["byear"] + 1).astype(int)

    # Step 2: Create regime code (group ID)
    df["regimecode"] = df.groupby("gwf_casename").ngroup()

    # Step 3: Expand to TSCS format (one row per year)
    df_expanded = df.loc[df.index.repeat(df["spell"])].copy()

    # Step 4: Create year variable (sequential within regime)
    df_expanded["year"] = (
        df_expanded.groupby("regimecode").cumcount() + df_expanded["byear"]
    ).astype(int)

    # Step 5: Create duration and failure variables
    df_expanded["duration"] = df_expanded["year"] - (df_expanded["byear"] - 1)
    df_expanded["failure"] = (
        (df_expanded["year"] == df_expanded["gwf_endyr"])
        & (df_expanded["gwf_endyr"].notna())
    ).astype(int)

    # Step 6: Create failure type variables
    df_expanded["gwf_fail"] = df_expanded["failure"]
    df_expanded["gwf_fail_type"] = 0.0
    df_expanded.loc[df_expanded["failure"] == 1, "gwf_fail_type"] = df_expanded.loc[
        df_expanded["failure"] == 1, "gwf_howend"
    ]

    df_expanded["gwf_fail_subsregime"] = df_expanded["gwf_fail"].astype(float)
    df_expanded["gwf_fail_subs"] = 0
    df_expanded.loc[df_expanded["failure"] == 1, "gwf_fail_subs"] = df_expanded.loc[
        df_expanded["failure"] == 1, "gwf_subsreg"
    ]

    df_expanded["gwf_fail_violent"] = df_expanded["gwf_fail"].astype(float)
    df_expanded.loc[df_expanded["failure"] == 1, "gwf_fail_violent"] = df_expanded.loc[
        df_expanded["failure"] == 1, "gwf_violent"
    ]

    # Step 7: Calculate past regime failures
    # Sort by cowcode and year
    df_expanded = df_expanded.sort_values(["cowcode", "year"]).reset_index(drop=True)

    # Calculate cumulative failures within country, then shift by 1 (lag)
    df_expanded["gwf_pastregimefail"] = (
        df_expanded.groupby("cowcode")["gwf_fail"].cumsum().shift(1).fillna(0).astype(int)
    )

    # Step 8: Create collapsed regime type dummies
    # gwf_party: party, party-military, party-personal, party-personal-military, oligarchy, Iran post-1980
    party_types = [
        "party",
        "party-military",
        "party-personal",
        "party-personal-military",
        "oligarchy",
    ]
    df_expanded["gwf_party"] = df_expanded["gwf_regimetype"].isin(party_types).astype(int)

    # Handle Iran post-1980 (cowcode 630)
    iran_post_1980 = (df_expanded["cowcode"] == 630) & (df_expanded["year"] > 1980)
    df_expanded.loc[iran_post_1980, "gwf_party"] = 1

    # gwf_personal: personal regime
    df_expanded["gwf_personal"] = (df_expanded["gwf_regimetype"] == "personal").astype(float)

    # gwf_monarch: monarchy
    df_expanded["gwf_monarch"] = (df_expanded["gwf_regimetype"] == "monarchy").astype(float)

    # gwf_military: military, military-personal, indirect military
    military_types = ["military", "military-personal", "indirect military"]
    df_expanded["gwf_military"] = df_expanded["gwf_regimetype"].isin(military_types).astype(int)

    # Step 9: Filter and select key variables
    df_tscs = df_expanded[df_expanded["year"] > 1945].copy()

    # Select key variables (21 columns from analysis plan)
    key_vars = [
        "cowcode",
        "year",
        "gwf_casename",
        "gwf_regimetype",
        "gwf_startyr",
        "gwf_endyr",
        "byear",
        "eyear",
        "spell",
        "duration",
        "failure",
        "gwf_fail",
        "gwf_fail_type",
        "gwf_fail_subsregime",
        "gwf_fail_subs",
        "gwf_fail_violent",
        "gwf_pastregimefail",
        "gwf_party",
        "gwf_personal",
        "gwf_monarch",
        "gwf_military",
    ]

    df_tscs = df_tscs[key_vars].reset_index(drop=True)

    return df_tscs

In [ ]:
data_dict = load_main_data()
gwf_tscs = run(data_dict['gwfcases'])
print(f"Max past failures: {int(gwf_tscs['gwf_pastregimefail'].max())}")
print(f"Mean past failures: {gwf_tscs['gwf_pastregimefail'].mean():.2f}")
print(f"Countries with 5+ past failures: {(gwf_tscs.groupby('cowcode')['gwf_pastregimefail'].max() >= 5).sum()}")
display(gwf_tscs[['cowcode', 'year', 'gwf_fail', 'gwf_pastregimefail']].head(20))


### Load GWF All Political Regimes

Load the comprehensive TSCS dataset containing all political regimes
(both autocracies and democracies) from 1946-2010. This is the base
dataset for Figure 1 analysis.

In [ ]:
def load_main_data() -> pd.DataFrame:
    """Load the main analysis datasets from converted data.

    This paper uses multiple datasets:
    - GWFcases: Regime case-level data (for clean.py transformation)
    - GWF_AllPoliticalRegimes: All political regimes TSCS data (for figure1.py)
    - institutions in dictatorships: Svolik data on authoritarian institutions (for figure1.py)

    Returns:
        Dictionary with keys 'gwfcases', 'gwf_all', 'svolik'.
    """
    # Load GWFcases (regime case-level data)
    gwfcases = pd.read_parquet(DATA_CONVERTED / "GWFcases.parquet")

    # Load GWF All Political Regimes (TSCS data)
    gwf_all = pd.read_parquet(DATA_CONVERTED / "GWF_AllPoliticalRegimes.parquet")

    # Load Svolik institutions data (1946-2008 version)
    svolik = pd.read_parquet(DATA_CONVERTED / "institutions in dictatorships, 1946-2008.parquet")

    return {
        "gwfcases": gwfcases,
        "gwf_all": gwf_all,
        "svolik": svolik,
    }

In [ ]:
data_dict = load_main_data()
gwf_all = data_dict['gwf_all']
print(f"GWF All shape: {gwf_all.shape}")
print(f"Countries: {gwf_all['cowcode'].nunique()}")
print(f"Year range: {gwf_all['year'].min()}-{gwf_all['year'].max()}")
print(f"Regime types: {gwf_all['gwf_regimetype'].value_counts()}")
print(f"Democracies: {(gwf_all['gwf_nonautocracy']=='democracy').sum()}")
display(gwf_all.head())


### Load Svolik Institutions Data

Load the Svolik (2012) authoritarian institutions dataset containing
information on executive selection, legislative characteristics, party
systems, and military characteristics for dictatorships.

In [ ]:
def load_main_data() -> pd.DataFrame:
    """Load the main analysis datasets from converted data.

    This paper uses multiple datasets:
    - GWFcases: Regime case-level data (for clean.py transformation)
    - GWF_AllPoliticalRegimes: All political regimes TSCS data (for figure1.py)
    - institutions in dictatorships: Svolik data on authoritarian institutions (for figure1.py)

    Returns:
        Dictionary with keys 'gwfcases', 'gwf_all', 'svolik'.
    """
    # Load GWFcases (regime case-level data)
    gwfcases = pd.read_parquet(DATA_CONVERTED / "GWFcases.parquet")

    # Load GWF All Political Regimes (TSCS data)
    gwf_all = pd.read_parquet(DATA_CONVERTED / "GWF_AllPoliticalRegimes.parquet")

    # Load Svolik institutions data (1946-2008 version)
    svolik = pd.read_parquet(DATA_CONVERTED / "institutions in dictatorships, 1946-2008.parquet")

    return {
        "gwfcases": gwfcases,
        "gwf_all": gwf_all,
        "svolik": svolik,
    }

In [ ]:
data_dict = load_main_data()
svolik = data_dict['svolik']
print(f"Svolik shape: {svolik.shape}")
print(f"Countries: {svolik['ccode'].nunique()}")
print(f"Year range: {svolik['year'].min()}-{svolik['year'].max()}")
print(f"Executive categories: {svolik['executive'].value_counts()}")
print(f"Legislative categories: {svolik['legislative'].value_counts()}")
display(svolik.head())


### Merge GWF and Svolik Data

Merge GWF All Political Regimes with Svolik institutions data using
1:1 merge on country code (ccode) and year. This combines regime type
classifications with institutional characteristics.

In [ ]:
def run(gwf_all: pd.DataFrame, svolik: pd.DataFrame) -> pd.DataFrame:
    """Generate Figure 1: Time trends in regime types.

    Args:
        gwf_all: GWF All Political Regimes DataFrame.
        svolik: Svolik institutions in dictatorships DataFrame.

    Returns:
        Year-level aggregated DataFrame with regime proportions.
    """
    ensure_output_dirs()

    # Step 1: Prepare GWF data (rename cowcode to ccode for merging)
    df = gwf_all.copy()
    df = df.rename(columns={"cowcode": "ccode"})

    # Step 2: Merge with Svolik data (1:1 on ccode and year)
    df = pd.merge(df, svolik, on=["ccode", "year"], how="left")

    # Step 3: Create regime category variables

    # democracy: 1 if gwf_nonautocracy == "democracy", 0 if gwf_regimetype != "NA"
    df["democracy"] = 0
    df.loc[df["gwf_nonautocracy"] == "democracy", "democracy"] = 1

    # dictatorship: 1 if gwf_regimetype != "NA", 0 if democracy == 1
    df["dictatorship"] = 0
    df.loc[df["gwf_regimetype"].notna(), "dictatorship"] = 1
    df.loc[df["democracy"] == 1, "dictatorship"] = 0

    # el_auth (electoral authoritarian):
    # 1 if executive is elected OR legislative has largest party controlling seats
    # 0 if dictatorship == 1 and not electoral
    # 0 if democracy == 1

    # First identify electoral criteria (from Svolik data)
    electoral = pd.Series(False, index=df.index)

    # Executive elected (any percentage)
    exec_elected = df["executive"].str.contains("elected", case=False, na=False)
    electoral |= exec_elected

    # Legislative has largest party controlling seats (any percentage)
    leg_largest = df["legislative"].str.contains("largest party controls", case=False, na=False)
    electoral |= leg_largest

    # Create el_auth variable
    df["el_auth"] = 0
    df.loc[(df["dictatorship"] == 1) & electoral, "el_auth"] = 1

    # nonel_auth (closed authoritarian):
    # 1 if el_auth == 0 and dictatorship == 1
    # 0 if el_auth == 1 or democracy == 1
    df["nonel_auth"] = 0
    df.loc[(df["dictatorship"] == 1) & (df["el_auth"] == 0), "nonel_auth"] = 1

    # Step 4: Aggregate to year-level proportions
    # Keep only years < 2009 (filter to 1946-2008)
    df_filtered = df[df["year"] < 2009].copy()

    # Collapse by year (mean of regime proportions)
    df_year = (
        df_filtered.groupby("year")[["el_auth", "nonel_auth", "democracy"]]
        .mean()
        .reset_index()
    )

    # Step 5: Generate time series plot
    fig, ax = plt.subplots(figsize=(10, 6))

    # Plot three lines
    ax.plot(df_year["year"], df_year["democracy"], label="Democracy", color="black", linewidth=2)
    ax.plot(
        df_year["year"],
        df_year["el_auth"],
        label="Electoral Authoritarian",
        color="red",
        linewidth=2,
    )
    ax.plot(
        df_year["year"],
        df_year["nonel_auth"],
        label="Closed Authoritarian",
        color="blue",
        linewidth=2,
    )

    # Axis labels and formatting
    ax.set_xlabel("Year", fontsize=12)
    ax.set_ylabel("Proportion of All Consolidated Regimes", fontsize=12)
    ax.set_xlim(1946, 2008)
    ax.set_ylim(0, 1)

    # Add legend
    ax.legend(loc="upper left", frameon=False)

    # Grid
    ax.grid(True, alpha=0.3)

    # Tight layout
    plt.tight_layout()

    # Save figure
    fig_path_pdf = FIGURE_DIR / "fig1.pdf"
    fig_path_png = FIGURE_DIR / "fig1.png"
    plt.savefig(fig_path_pdf, dpi=300, bbox_inches="tight")
    plt.savefig(fig_path_png, dpi=300, bbox_inches="tight")
    plt.close()

    print(f"Figure saved to {fig_path_pdf} and {fig_path_png}")

    return df_year

In [ ]:
data_dict = load_main_data()
df_year = run(data_dict['gwf_all'], data_dict['svolik'])
print(f"Merged data points: {len(df_year)}")
print(f"Year range: {df_year['year'].min()}-{df_year['year'].max()}")
display(df_year.head(10))


### Create Regime Categories

Create three mutually exclusive regime categories: democracy (GWF coded),
electoral authoritarian (dictatorship with elected executive OR legislature
with parties), and closed authoritarian (dictatorship without elections).

In [ ]:
def run(gwf_all: pd.DataFrame, svolik: pd.DataFrame) -> pd.DataFrame:
    """Generate Figure 1: Time trends in regime types.

    Args:
        gwf_all: GWF All Political Regimes DataFrame.
        svolik: Svolik institutions in dictatorships DataFrame.

    Returns:
        Year-level aggregated DataFrame with regime proportions.
    """
    ensure_output_dirs()

    # Step 1: Prepare GWF data (rename cowcode to ccode for merging)
    df = gwf_all.copy()
    df = df.rename(columns={"cowcode": "ccode"})

    # Step 2: Merge with Svolik data (1:1 on ccode and year)
    df = pd.merge(df, svolik, on=["ccode", "year"], how="left")

    # Step 3: Create regime category variables

    # democracy: 1 if gwf_nonautocracy == "democracy", 0 if gwf_regimetype != "NA"
    df["democracy"] = 0
    df.loc[df["gwf_nonautocracy"] == "democracy", "democracy"] = 1

    # dictatorship: 1 if gwf_regimetype != "NA", 0 if democracy == 1
    df["dictatorship"] = 0
    df.loc[df["gwf_regimetype"].notna(), "dictatorship"] = 1
    df.loc[df["democracy"] == 1, "dictatorship"] = 0

    # el_auth (electoral authoritarian):
    # 1 if executive is elected OR legislative has largest party controlling seats
    # 0 if dictatorship == 1 and not electoral
    # 0 if democracy == 1

    # First identify electoral criteria (from Svolik data)
    electoral = pd.Series(False, index=df.index)

    # Executive elected (any percentage)
    exec_elected = df["executive"].str.contains("elected", case=False, na=False)
    electoral |= exec_elected

    # Legislative has largest party controlling seats (any percentage)
    leg_largest = df["legislative"].str.contains("largest party controls", case=False, na=False)
    electoral |= leg_largest

    # Create el_auth variable
    df["el_auth"] = 0
    df.loc[(df["dictatorship"] == 1) & electoral, "el_auth"] = 1

    # nonel_auth (closed authoritarian):
    # 1 if el_auth == 0 and dictatorship == 1
    # 0 if el_auth == 1 or democracy == 1
    df["nonel_auth"] = 0
    df.loc[(df["dictatorship"] == 1) & (df["el_auth"] == 0), "nonel_auth"] = 1

    # Step 4: Aggregate to year-level proportions
    # Keep only years < 2009 (filter to 1946-2008)
    df_filtered = df[df["year"] < 2009].copy()

    # Collapse by year (mean of regime proportions)
    df_year = (
        df_filtered.groupby("year")[["el_auth", "nonel_auth", "democracy"]]
        .mean()
        .reset_index()
    )

    # Step 5: Generate time series plot
    fig, ax = plt.subplots(figsize=(10, 6))

    # Plot three lines
    ax.plot(df_year["year"], df_year["democracy"], label="Democracy", color="black", linewidth=2)
    ax.plot(
        df_year["year"],
        df_year["el_auth"],
        label="Electoral Authoritarian",
        color="red",
        linewidth=2,
    )
    ax.plot(
        df_year["year"],
        df_year["nonel_auth"],
        label="Closed Authoritarian",
        color="blue",
        linewidth=2,
    )

    # Axis labels and formatting
    ax.set_xlabel("Year", fontsize=12)
    ax.set_ylabel("Proportion of All Consolidated Regimes", fontsize=12)
    ax.set_xlim(1946, 2008)
    ax.set_ylim(0, 1)

    # Add legend
    ax.legend(loc="upper left", frameon=False)

    # Grid
    ax.grid(True, alpha=0.3)

    # Tight layout
    plt.tight_layout()

    # Save figure
    fig_path_pdf = FIGURE_DIR / "fig1.pdf"
    fig_path_png = FIGURE_DIR / "fig1.png"
    plt.savefig(fig_path_pdf, dpi=300, bbox_inches="tight")
    plt.savefig(fig_path_png, dpi=300, bbox_inches="tight")
    plt.close()

    print(f"Figure saved to {fig_path_pdf} and {fig_path_png}")

    return df_year

In [ ]:
data_dict = load_main_data()
df_year = run(data_dict['gwf_all'], data_dict['svolik'])
print(f"Mean democracy: {df_year['democracy'].mean():.3f}")
print(f"Mean electoral auth: {df_year['el_auth'].mean():.3f}")
print(f"Mean closed auth: {df_year['nonel_auth'].mean():.3f}")
print(f"Sum check (should be ~1.0): {(df_year['democracy'] + df_year['el_auth'] + df_year['nonel_auth']).mean():.3f}")
display(df_year)


### Aggregate to Year-Level Proportions

Collapse the country-year data to year-level means, calculating the
proportion of all consolidated regimes that are democratic, electoral
authoritarian, or closed authoritarian in each year (1946-2008).

In [ ]:
def run(gwf_all: pd.DataFrame, svolik: pd.DataFrame) -> pd.DataFrame:
    """Generate Figure 1: Time trends in regime types.

    Args:
        gwf_all: GWF All Political Regimes DataFrame.
        svolik: Svolik institutions in dictatorships DataFrame.

    Returns:
        Year-level aggregated DataFrame with regime proportions.
    """
    ensure_output_dirs()

    # Step 1: Prepare GWF data (rename cowcode to ccode for merging)
    df = gwf_all.copy()
    df = df.rename(columns={"cowcode": "ccode"})

    # Step 2: Merge with Svolik data (1:1 on ccode and year)
    df = pd.merge(df, svolik, on=["ccode", "year"], how="left")

    # Step 3: Create regime category variables

    # democracy: 1 if gwf_nonautocracy == "democracy", 0 if gwf_regimetype != "NA"
    df["democracy"] = 0
    df.loc[df["gwf_nonautocracy"] == "democracy", "democracy"] = 1

    # dictatorship: 1 if gwf_regimetype != "NA", 0 if democracy == 1
    df["dictatorship"] = 0
    df.loc[df["gwf_regimetype"].notna(), "dictatorship"] = 1
    df.loc[df["democracy"] == 1, "dictatorship"] = 0

    # el_auth (electoral authoritarian):
    # 1 if executive is elected OR legislative has largest party controlling seats
    # 0 if dictatorship == 1 and not electoral
    # 0 if democracy == 1

    # First identify electoral criteria (from Svolik data)
    electoral = pd.Series(False, index=df.index)

    # Executive elected (any percentage)
    exec_elected = df["executive"].str.contains("elected", case=False, na=False)
    electoral |= exec_elected

    # Legislative has largest party controlling seats (any percentage)
    leg_largest = df["legislative"].str.contains("largest party controls", case=False, na=False)
    electoral |= leg_largest

    # Create el_auth variable
    df["el_auth"] = 0
    df.loc[(df["dictatorship"] == 1) & electoral, "el_auth"] = 1

    # nonel_auth (closed authoritarian):
    # 1 if el_auth == 0 and dictatorship == 1
    # 0 if el_auth == 1 or democracy == 1
    df["nonel_auth"] = 0
    df.loc[(df["dictatorship"] == 1) & (df["el_auth"] == 0), "nonel_auth"] = 1

    # Step 4: Aggregate to year-level proportions
    # Keep only years < 2009 (filter to 1946-2008)
    df_filtered = df[df["year"] < 2009].copy()

    # Collapse by year (mean of regime proportions)
    df_year = (
        df_filtered.groupby("year")[["el_auth", "nonel_auth", "democracy"]]
        .mean()
        .reset_index()
    )

    # Step 5: Generate time series plot
    fig, ax = plt.subplots(figsize=(10, 6))

    # Plot three lines
    ax.plot(df_year["year"], df_year["democracy"], label="Democracy", color="black", linewidth=2)
    ax.plot(
        df_year["year"],
        df_year["el_auth"],
        label="Electoral Authoritarian",
        color="red",
        linewidth=2,
    )
    ax.plot(
        df_year["year"],
        df_year["nonel_auth"],
        label="Closed Authoritarian",
        color="blue",
        linewidth=2,
    )

    # Axis labels and formatting
    ax.set_xlabel("Year", fontsize=12)
    ax.set_ylabel("Proportion of All Consolidated Regimes", fontsize=12)
    ax.set_xlim(1946, 2008)
    ax.set_ylim(0, 1)

    # Add legend
    ax.legend(loc="upper left", frameon=False)

    # Grid
    ax.grid(True, alpha=0.3)

    # Tight layout
    plt.tight_layout()

    # Save figure
    fig_path_pdf = FIGURE_DIR / "fig1.pdf"
    fig_path_png = FIGURE_DIR / "fig1.png"
    plt.savefig(fig_path_pdf, dpi=300, bbox_inches="tight")
    plt.savefig(fig_path_png, dpi=300, bbox_inches="tight")
    plt.close()

    print(f"Figure saved to {fig_path_pdf} and {fig_path_png}")

    return df_year

In [ ]:
data_dict = load_main_data()
df_year = run(data_dict['gwf_all'], data_dict['svolik'])
print(f"Years: {len(df_year)}")
print(f"Year range: {df_year['year'].min()}-{df_year['year'].max()}")
print("\nFirst 10 years:")
display(df_year.head(10))
print("\nLast 10 years:")
display(df_year.tail(10))


## Data Cleaning


### Data Cleaning

Transform GWFcases to time-series cross-sectional (TSCS) format with regime failures and regime type dummies (implements clean.do)

In [ ]:
def run(gwfcases: pd.DataFrame) -> pd.DataFrame:
    """Transform GWFcases to time-series cross-sectional (TSCS) format.

    Args:
        gwfcases: Regime case-level DataFrame from load_main_data().

    Returns:
        GWFtscs DataFrame with one row per country-year.
    """
    df = gwfcases.copy()

    # Step 1: Create temporal variables
    df["byear"] = df["gwf_startyr"]
    df["eyear"] = df["gwf_endyr"].fillna(2010)  # Right-censoring
    df["spell"] = (df["eyear"] - df["byear"] + 1).astype(int)

    # Step 2: Create regime code (group ID)
    df["regimecode"] = df.groupby("gwf_casename").ngroup()

    # Step 3: Expand to TSCS format (one row per year)
    df_expanded = df.loc[df.index.repeat(df["spell"])].copy()

    # Step 4: Create year variable (sequential within regime)
    df_expanded["year"] = (
        df_expanded.groupby("regimecode").cumcount() + df_expanded["byear"]
    ).astype(int)

    # Step 5: Create duration and failure variables
    df_expanded["duration"] = df_expanded["year"] - (df_expanded["byear"] - 1)
    df_expanded["failure"] = (
        (df_expanded["year"] == df_expanded["gwf_endyr"])
        & (df_expanded["gwf_endyr"].notna())
    ).astype(int)

    # Step 6: Create failure type variables
    df_expanded["gwf_fail"] = df_expanded["failure"]
    df_expanded["gwf_fail_type"] = 0.0
    df_expanded.loc[df_expanded["failure"] == 1, "gwf_fail_type"] = df_expanded.loc[
        df_expanded["failure"] == 1, "gwf_howend"
    ]

    df_expanded["gwf_fail_subsregime"] = df_expanded["gwf_fail"].astype(float)
    df_expanded["gwf_fail_subs"] = 0
    df_expanded.loc[df_expanded["failure"] == 1, "gwf_fail_subs"] = df_expanded.loc[
        df_expanded["failure"] == 1, "gwf_subsreg"
    ]

    df_expanded["gwf_fail_violent"] = df_expanded["gwf_fail"].astype(float)
    df_expanded.loc[df_expanded["failure"] == 1, "gwf_fail_violent"] = df_expanded.loc[
        df_expanded["failure"] == 1, "gwf_violent"
    ]

    # Step 7: Calculate past regime failures
    # Sort by cowcode and year
    df_expanded = df_expanded.sort_values(["cowcode", "year"]).reset_index(drop=True)

    # Calculate cumulative failures within country, then shift by 1 (lag)
    df_expanded["gwf_pastregimefail"] = (
        df_expanded.groupby("cowcode")["gwf_fail"].cumsum().shift(1).fillna(0).astype(int)
    )

    # Step 8: Create collapsed regime type dummies
    # gwf_party: party, party-military, party-personal, party-personal-military, oligarchy, Iran post-1980
    party_types = [
        "party",
        "party-military",
        "party-personal",
        "party-personal-military",
        "oligarchy",
    ]
    df_expanded["gwf_party"] = df_expanded["gwf_regimetype"].isin(party_types).astype(int)

    # Handle Iran post-1980 (cowcode 630)
    iran_post_1980 = (df_expanded["cowcode"] == 630) & (df_expanded["year"] > 1980)
    df_expanded.loc[iran_post_1980, "gwf_party"] = 1

    # gwf_personal: personal regime
    df_expanded["gwf_personal"] = (df_expanded["gwf_regimetype"] == "personal").astype(float)

    # gwf_monarch: monarchy
    df_expanded["gwf_monarch"] = (df_expanded["gwf_regimetype"] == "monarchy").astype(float)

    # gwf_military: military, military-personal, indirect military
    military_types = ["military", "military-personal", "indirect military"]
    df_expanded["gwf_military"] = df_expanded["gwf_regimetype"].isin(military_types).astype(int)

    # Step 9: Filter and select key variables
    df_tscs = df_expanded[df_expanded["year"] > 1945].copy()

    # Select key variables (21 columns from analysis plan)
    key_vars = [
        "cowcode",
        "year",
        "gwf_casename",
        "gwf_regimetype",
        "gwf_startyr",
        "gwf_endyr",
        "byear",
        "eyear",
        "spell",
        "duration",
        "failure",
        "gwf_fail",
        "gwf_fail_type",
        "gwf_fail_subsregime",
        "gwf_fail_subs",
        "gwf_fail_violent",
        "gwf_pastregimefail",
        "gwf_party",
        "gwf_personal",
        "gwf_monarch",
        "gwf_military",
    ]

    df_tscs = df_tscs[key_vars].reset_index(drop=True)

    return df_tscs

In [ ]:
result = run()
print(type(result))
if isinstance(result, dict):
    print(list(result.keys()))

## Figure 1: Electoralism, 1946-2008


### Figure 1: Electoralism, 1946-2008

Time trends in regime types: democratic, electoral authoritarian, and closed authoritarian regimes (implements figure1.do)

In [ ]:
def run(gwf_all: pd.DataFrame, svolik: pd.DataFrame) -> pd.DataFrame:
    """Generate Figure 1: Time trends in regime types.

    Args:
        gwf_all: GWF All Political Regimes DataFrame.
        svolik: Svolik institutions in dictatorships DataFrame.

    Returns:
        Year-level aggregated DataFrame with regime proportions.
    """
    ensure_output_dirs()

    # Step 1: Prepare GWF data (rename cowcode to ccode for merging)
    df = gwf_all.copy()
    df = df.rename(columns={"cowcode": "ccode"})

    # Step 2: Merge with Svolik data (1:1 on ccode and year)
    df = pd.merge(df, svolik, on=["ccode", "year"], how="left")

    # Step 3: Create regime category variables

    # democracy: 1 if gwf_nonautocracy == "democracy", 0 if gwf_regimetype != "NA"
    df["democracy"] = 0
    df.loc[df["gwf_nonautocracy"] == "democracy", "democracy"] = 1

    # dictatorship: 1 if gwf_regimetype != "NA", 0 if democracy == 1
    df["dictatorship"] = 0
    df.loc[df["gwf_regimetype"].notna(), "dictatorship"] = 1
    df.loc[df["democracy"] == 1, "dictatorship"] = 0

    # el_auth (electoral authoritarian):
    # 1 if executive is elected OR legislative has largest party controlling seats
    # 0 if dictatorship == 1 and not electoral
    # 0 if democracy == 1

    # First identify electoral criteria (from Svolik data)
    electoral = pd.Series(False, index=df.index)

    # Executive elected (any percentage)
    exec_elected = df["executive"].str.contains("elected", case=False, na=False)
    electoral |= exec_elected

    # Legislative has largest party controlling seats (any percentage)
    leg_largest = df["legislative"].str.contains("largest party controls", case=False, na=False)
    electoral |= leg_largest

    # Create el_auth variable
    df["el_auth"] = 0
    df.loc[(df["dictatorship"] == 1) & electoral, "el_auth"] = 1

    # nonel_auth (closed authoritarian):
    # 1 if el_auth == 0 and dictatorship == 1
    # 0 if el_auth == 1 or democracy == 1
    df["nonel_auth"] = 0
    df.loc[(df["dictatorship"] == 1) & (df["el_auth"] == 0), "nonel_auth"] = 1

    # Step 4: Aggregate to year-level proportions
    # Keep only years < 2009 (filter to 1946-2008)
    df_filtered = df[df["year"] < 2009].copy()

    # Collapse by year (mean of regime proportions)
    df_year = (
        df_filtered.groupby("year")[["el_auth", "nonel_auth", "democracy"]]
        .mean()
        .reset_index()
    )

    # Step 5: Generate time series plot
    fig, ax = plt.subplots(figsize=(10, 6))

    # Plot three lines
    ax.plot(df_year["year"], df_year["democracy"], label="Democracy", color="black", linewidth=2)
    ax.plot(
        df_year["year"],
        df_year["el_auth"],
        label="Electoral Authoritarian",
        color="red",
        linewidth=2,
    )
    ax.plot(
        df_year["year"],
        df_year["nonel_auth"],
        label="Closed Authoritarian",
        color="blue",
        linewidth=2,
    )

    # Axis labels and formatting
    ax.set_xlabel("Year", fontsize=12)
    ax.set_ylabel("Proportion of All Consolidated Regimes", fontsize=12)
    ax.set_xlim(1946, 2008)
    ax.set_ylim(0, 1)

    # Add legend
    ax.legend(loc="upper left", frameon=False)

    # Grid
    ax.grid(True, alpha=0.3)

    # Tight layout
    plt.tight_layout()

    # Save figure
    fig_path_pdf = FIGURE_DIR / "fig1.pdf"
    fig_path_png = FIGURE_DIR / "fig1.png"
    plt.savefig(fig_path_pdf, dpi=300, bbox_inches="tight")
    plt.savefig(fig_path_png, dpi=300, bbox_inches="tight")
    plt.close()

    print(f"Figure saved to {fig_path_pdf} and {fig_path_png}")

    return df_year

In [ ]:
result = run()
print(type(result))
if isinstance(result, dict):
    print(list(result.keys()))